# 🧬 Hybrid CNN + PCA + Quantum VQC for Oral Cancer Detection

---

## 📌 Overview

This notebook implements a **Hybrid Quantum-Classical Deep Learning pipeline** for binary classification of oral cancer images.

### Architecture
```
Input Image
    ↓
EfficientNet-B3 (Pretrained CNN Backbone — Feature Extractor)
    ↓
    ├──→ Classical MLP (Full 1536-dim features)
    │
    └──→ PCA (1536-dim → 10-dim) + MinMaxScaler
              ↓
         Quantum VQC (10 qubits, 4 layers via PennyLane)
              ↓
         Fully-connected Classifier
    ↓
Weighted Ensemble (MLP 60% + VQC 40%)
    ↓
Cancer / No Cancer
```

### Key Highlights
- 🔬 **Quantum Circuit**: Variational Quantum Classifier (VQC) with `Rot` gates + `CNOT` entanglement using **PennyLane**
- 🧠 **Classical Backbone**: EfficientNet-B3 pretrained on ImageNet for rich feature extraction
- ⚡ **Dimensionality Reduction**: PCA reduces 1536-dim CNN features to 10 quantum-compatible features
- 🎯 **Ensemble Strategy**: Weighted voting (MLP + VQC) for maximum accuracy

### Dataset
- Binary classification: `Cancer` vs `Non-Cancer` oral cavity images
- Split: 70% Train | 15% Validation | 15% Test
- Class imbalance handled via `WeightedRandomSampler`

---

> **Author:** [Your Name]  
> **Framework:** PyTorch + PennyLane  
> **Hardware:** GPU (CUDA) / CPU fallback

## 1️⃣ Install Dependencies

In [ ]:
!pip install pennylane torch torchvision scikit-learn tqdm -q

## 2️⃣ Mount Google Drive

Mount your Google Drive to access the oral cancer image dataset.

> **Expected folder structure:**
> ```
> /content/drive/MyDrive/cancer_detection/oral/
> ├── cancer/
> │   ├── img1.jpg
> │   └── ...
> └── non_cancer/
>     ├── img1.jpg
>     └── ...
> ```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3️⃣ Imports & Configuration

In [ ]:
import os, random, numpy as np, torch, torch.nn as nn, torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader, random_split, WeightedRandomSampler
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, roc_auc_score
from tqdm import tqdm
import pennylane as qml
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter

# ── Reproducibility ──────────────────────────────────────────
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()
torch.set_default_dtype(torch.float32)

# ── Hardware ─────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {DEVICE}")

# ── Hyperparameters ───────────────────────────────────────────
DATA_DIR         = "/content/drive/MyDrive/cancer_detection/oral"
IMG_SIZE         = 224     # EfficientNet-B3 input size
BATCH_SIZE       = 16
NUM_WORKERS      = 2
PCA_DIM          = 10      # Reduced dims fed into quantum circuit
VQC_LAYERS       = 4       # Number of variational layers in quantum circuit
EPOCHS_CLASSICAL = 40
EPOCHS_VQC       = 35
LR               = 1e-3
NUM_CLASSES      = 2       # Cancer | Non-Cancer

## 4️⃣ Data Loading & Augmentation

- **Training**: Heavy augmentation (flips, rotation, color jitter) to improve generalization
- **Validation/Test**: Only resize + normalize (no augmentation to prevent data leakage)
- **Class imbalance**: Handled using `WeightedRandomSampler` on the training set

In [ ]:
# ── Image Transforms ─────────────────────────────────────────
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # ImageNet stats
])

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ── Load Dataset ─────────────────────────────────────────────
full_dataset = datasets.ImageFolder(DATA_DIR, transform=train_tf)
class_names  = list(full_dataset.class_to_idx.keys())
print(f"📂 Classes found : {class_names}")
print(f"📊 Total samples : {len(full_dataset)}")

# ── Train / Val / Test Split (70/15/15) ───────────────────────
train_size = int(0.70 * len(full_dataset))
val_size   = int(0.15 * len(full_dataset))
test_size  = len(full_dataset) - train_size - val_size

train_ds, val_ds, test_ds = random_split(
    full_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(SEED)
)

# Apply eval transforms to val/test (no augmentation)
val_ds.dataset.transform  = eval_tf
test_ds.dataset.transform = eval_tf

print(f"\n🔀 Split → Train: {train_size} | Val: {val_size} | Test: {test_size}")

# ── Weighted Sampler for class imbalance ─────────────────────
train_labels = [full_dataset.targets[i] for i in train_ds.indices]
weights      = [1.0 / Counter(train_labels)[l] for l in train_labels]
sampler      = WeightedRandomSampler(weights, len(weights))

# ── DataLoaders ───────────────────────────────────────────────
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print("✅ DataLoaders ready")

## 5️⃣ Feature Extraction with EfficientNet-B3

We use **EfficientNet-B3** (pretrained on ImageNet) as a frozen feature extractor.
- The classifier head is replaced with `nn.Identity()` to extract raw feature vectors
- Output: `1536-dimensional` feature vectors per image
- No fine-tuning — all gradients are frozen (`requires_grad = False`)

In [ ]:
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights

print("⚙️  Loading EfficientNet-B3 backbone...")
backbone = efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT)
backbone.classifier = nn.Identity()   # Remove final classification head
backbone = backbone.to(DEVICE)
backbone.eval()

# Freeze all backbone parameters
for p in backbone.parameters():
    p.requires_grad = False

print("✅ Backbone loaded and frozen")

def extract_features(loader, desc="Extracting"):
    """Pass all images through the frozen backbone and collect feature vectors."""
    feats, labels = [], []
    with torch.no_grad():
        for x, y in tqdm(loader, desc=desc):
            feats.append(backbone(x.to(DEVICE)).cpu().numpy())
            labels.append(y.numpy())
    return np.vstack(feats), np.concatenate(labels)

print("\n🔍 Extracting features from all splits...")
train_feat, train_y = extract_features(train_loader, "Train features")
val_feat,   val_y   = extract_features(val_loader,   "Val features  ")
test_feat,  test_y  = extract_features(test_loader,  "Test features ")

print(f"\n📐 Feature shape → Train: {train_feat.shape} | Val: {val_feat.shape} | Test: {test_feat.shape}")

## 6️⃣ Data Preparation — Classical & Quantum Branches

The pipeline **splits into two branches** after feature extraction:

| Branch | Input | Preprocessing |
|--------|-------|----------------|
| Classical MLP | Raw 1536-dim features | None (full features) |
| Quantum VQC | 1536-dim features | PCA → 10 dims, MinMaxScaler → [-1, 1] |

In [ ]:
# ── Classical branch: raw features as tensors ────────────────
X_train_mlp = torch.tensor(train_feat, dtype=torch.float32).to(DEVICE)
X_val_mlp   = torch.tensor(val_feat,   dtype=torch.float32).to(DEVICE)
X_test_mlp  = torch.tensor(test_feat,  dtype=torch.float32).to(DEVICE)

y_train_t = torch.tensor(train_y, dtype=torch.long).to(DEVICE)
y_val_t   = torch.tensor(val_y,   dtype=torch.long).to(DEVICE)

# ── Quantum branch: PCA + MinMaxScaler ───────────────────────
# PCA reduces 1536-dim → 10-dim (matches number of qubits)
# MinMaxScaler maps features to [-1, 1] for stable RY/RZ rotation angles
pca    = PCA(n_components=PCA_DIM, random_state=SEED)
scaler = MinMaxScaler(feature_range=(-1, 1))

X_train_vqc = torch.tensor(
    scaler.fit_transform(pca.fit_transform(train_feat)), dtype=torch.float32).to(DEVICE)
X_val_vqc   = torch.tensor(
    scaler.transform(pca.transform(val_feat)), dtype=torch.float32).to(DEVICE)
X_test_vqc  = torch.tensor(
    scaler.transform(pca.transform(test_feat)), dtype=torch.float32).to(DEVICE)

print(f"✅ Classical branch shape : {X_train_mlp.shape}")
print(f"✅ Quantum branch shape   : {X_train_vqc.shape}")
print(f"   PCA explained variance : {pca.explained_variance_ratio_.sum():.3f}")

## 7️⃣ Classical MLP Training

A 3-layer MLP trained on full EfficientNet-B3 features:

```
Linear(1536 → 512) → BatchNorm → GELU → Dropout(0.3)
Linear(512 → 128)  → BatchNorm → GELU → Dropout(0.2)
Linear(128 → 2)    → Output
```

**Training tricks used:**
- `AdamW` optimizer with weight decay
- `CosineAnnealingLR` scheduler
- `CrossEntropyLoss` with label smoothing (0.1) to prevent overconfidence
- Best model checkpoint saved based on validation accuracy

In [ ]:
class ImprovedMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512), nn.BatchNorm1d(512), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(512, 128),       nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(128, NUM_CLASSES)
        )
    def forward(self, x):
        return self.net(x)

# ── Model, Optimizer, Loss, Scheduler ────────────────────────
mlp           = ImprovedMLP(train_feat.shape[1]).to(DEVICE)
optimizer     = optim.AdamW(mlp.parameters(), lr=LR, weight_decay=1e-3)
loss_fn       = nn.CrossEntropyLoss(label_smoothing=0.1)
scheduler_mlp = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_CLASSICAL)

best_mlp_acc   = 0
best_mlp_state = None

print("🧠 Training Classical MLP...")
print("-" * 50)

for epoch in range(EPOCHS_CLASSICAL):
    mlp.train()
    optimizer.zero_grad()
    loss = loss_fn(mlp(X_train_mlp), y_train_t)
    loss.backward()
    optimizer.step()
    scheduler_mlp.step()

    mlp.eval()
    with torch.no_grad():
        val_preds = torch.argmax(mlp(X_val_mlp), dim=1)
        val_acc   = accuracy_score(y_val_t.cpu(), val_preds.cpu())

    # Save best checkpoint
    if val_acc > best_mlp_acc:
        best_mlp_acc   = val_acc
        best_mlp_state = {k: v.clone() for k, v in mlp.state_dict().items()}

    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1:>3}/{EPOCHS_CLASSICAL} | Loss: {loss.item():.4f} | Val Acc: {val_acc:.4f}")

# Restore best checkpoint
mlp.load_state_dict(best_mlp_state)
mlp.eval()

with torch.no_grad():
    mlp_probs_test = torch.softmax(mlp(X_test_mlp), dim=1).cpu().numpy()
    mlp_preds      = np.argmax(mlp_probs_test, axis=1)

print(f"\n✅ MLP training done | Best Val Acc: {best_mlp_acc:.4f}")

## 8️⃣ Quantum VQC Training

### Circuit Design
- **10 qubits** (matches PCA output dimension)
- **4 variational layers**, each containing:
  - `RY + RZ` encoding gates per qubit (input encoding)
  - `Rot` gates per qubit (trainable rotation: 3 params/qubit/layer)
  - `CNOT` ring entanglement (linear chain + wraparound)
- **Output**: Expectation values `⟨Z⟩` for all 10 qubits → fed into FC classifier

```
x[i] → RY(x[i]·π) → RZ(x[i]·π/2) → Rot(θ,φ,λ) ─┐
                                                   CNOT chain
                                                    ↓
                                             Measure ⟨Z⟩ × 10
                                                    ↓
                                            FC(10→16→2)
```

In [ ]:
n_qubits = PCA_DIM   # 10 qubits
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch", diff_method="backprop")
def quantum_circuit(inputs, weights):
    """
    Variational Quantum Circuit (VQC).

    Args:
        inputs  : PCA-reduced + scaled feature vector  [n_qubits]
        weights : Trainable rotation parameters        [VQC_LAYERS, n_qubits, 3]

    Returns:
        List of PauliZ expectation values for each qubit.
    """
    # ── Data Encoding ────────────────────────────────────────
    for i in range(n_qubits):
        qml.RY(inputs[i] * np.pi,     wires=i)   # Amplitude encoding
        qml.RZ(inputs[i] * np.pi / 2, wires=i)   # Phase encoding

    # ── Variational Layers ───────────────────────────────────
    for l in range(weights.shape[0]):
        # Trainable Rot gates (3 params: phi, theta, omega)
        for i in range(n_qubits):
            qml.Rot(weights[l, i, 0], weights[l, i, 1], weights[l, i, 2], wires=i)

        # CNOT entanglement ring (creates correlations between qubits)
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i + 1])
        qml.CNOT(wires=[n_qubits - 1, 0])   # Close the ring

    # ── Measurement ──────────────────────────────────────────
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]


class ImprovedVQC(nn.Module):
    def __init__(self):
        super().__init__()
        # Trainable quantum weights: [layers, qubits, 3 rotation params]
        self.weights = nn.Parameter(torch.randn(VQC_LAYERS, n_qubits, 3) * 0.1)
        # Classical post-processing layer
        self.fc = nn.Sequential(
            nn.Linear(n_qubits, 16), nn.GELU(),
            nn.Linear(16, NUM_CLASSES)
        )

    def forward(self, x):
        if x.dim() == 1:
            x = x.unsqueeze(0)
        # Run each sample through the quantum circuit
        outputs = []
        for i in range(x.shape[0]):
            outputs.append(torch.stack(list(quantum_circuit(x[i], self.weights))))
        return self.fc(torch.stack(outputs).float())


# Print quantum circuit structure
print("⚛️  Quantum Circuit Summary")
print(f"   Qubits   : {n_qubits}")
print(f"   Layers   : {VQC_LAYERS}")
print(f"   Params   : {VQC_LAYERS * n_qubits * 3} trainable quantum params")
print(f"   Backend  : PennyLane default.qubit (simulator)")

In [ ]:
vqc           = ImprovedVQC().to(DEVICE)
vqc_optimizer = optim.AdamW(vqc.parameters(), lr=2e-3, weight_decay=1e-3)
scheduler_vqc = optim.lr_scheduler.CosineAnnealingLR(vqc_optimizer, T_max=EPOCHS_VQC)

vqc_train_loader = DataLoader(
    torch.utils.data.TensorDataset(X_train_vqc, y_train_t),
    batch_size=8, shuffle=True
)

best_vqc_acc   = 0
best_vqc_state = None

print("⚛️  Training Quantum VQC...")
print("-" * 50)

for epoch in range(EPOCHS_VQC):
    vqc.train()
    total_loss = 0
    for xb, yb in tqdm(vqc_train_loader, desc=f"Epoch {epoch+1}/{EPOCHS_VQC}", leave=False):
        vqc_optimizer.zero_grad()
        loss = loss_fn(vqc(xb), yb)
        loss.backward()
        vqc_optimizer.step()
        total_loss += loss.item()

    scheduler_vqc.step()

    vqc.eval()
    with torch.no_grad():
        vqc_val_preds = torch.argmax(vqc(X_val_vqc), dim=1).cpu().numpy()
        val_acc       = accuracy_score(val_y, vqc_val_preds)

    if val_acc > best_vqc_acc:
        best_vqc_acc   = val_acc
        best_vqc_state = {k: v.clone() for k, v in vqc.state_dict().items()}

    print(f"  Epoch {epoch+1:>3}/{EPOCHS_VQC} | Loss: {total_loss:.4f} | Val Acc: {val_acc:.4f}")

# Restore best checkpoint
vqc.load_state_dict(best_vqc_state)
vqc.eval()

with torch.no_grad():
    vqc_probs_test = torch.softmax(vqc(X_test_vqc), dim=1).cpu().numpy()
    vqc_preds      = np.argmax(vqc_probs_test, axis=1)

print(f"\n✅ VQC training done | Best Val Acc: {best_vqc_acc:.4f}")

## 9️⃣ Ensemble & Results

**Weighted ensemble** combines both model predictions:
- MLP contributes **60%** (stronger on high-dim features)
- VQC contributes **40%** (captures quantum feature interactions)

Final prediction = `argmax(0.6 × MLP_probs + 0.4 × VQC_probs)`

In [ ]:
# ── Weighted Ensemble ────────────────────────────────────────
ensemble_probs = (mlp_probs_test * 0.6) + (vqc_probs_test * 0.4)
ensemble_preds = np.argmax(ensemble_probs, axis=1)

# ── Accuracy Comparison ──────────────────────────────────────
mlp_acc = accuracy_score(test_y, mlp_preds)
vqc_acc = accuracy_score(test_y, vqc_preds)
ens_acc = accuracy_score(test_y, ensemble_preds)

print("\n" + "="*60)
print(f"{'Model':<20} {'Accuracy':>10}")
print("-"*60)
print(f"{'Classical MLP':<20} {mlp_acc:>10.4f}")
print(f"{'Quantum VQC':<20} {vqc_acc:>10.4f}")
print(f"{'Ensemble (Final)':<20} {ens_acc:>10.4f}  ← Final Model")
print("="*60)

# ── Detailed Classification Report ───────────────────────────
from sklearn.metrics import classification_report
print("\n📋 Classification Report (Ensemble):")
print(classification_report(test_y, ensemble_preds, target_names=class_names))

# ── Confusion Matrix ─────────────────────────────────────────
cm = confusion_matrix(test_y, ensemble_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix — Ensemble Model', fontsize=13, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

# ── ROC-AUC ──────────────────────────────────────────────────
try:
    auc = roc_auc_score(test_y, ensemble_probs[:, 1])
    print(f"\n📈 ROC-AUC Score (Ensemble): {auc:.4f}")
except Exception as e:
    print(f"ROC-AUC could not be computed: {e}")

## 🔟 Single Image Inference

Upload one or more images to get a prediction from the ensemble model.
- Both MLP and VQC predictions are shown individually
- Final verdict comes from the weighted ensemble

In [ ]:
def predict_image(image_path):
    """Run inference on a single image using the trained ensemble."""
    img        = Image.open(image_path).convert('RGB')
    img_tensor = eval_tf(img).unsqueeze(0).to(DEVICE)

    # Extract features using backbone
    with torch.no_grad():
        feat = backbone(img_tensor).cpu().numpy()

    # Prepare inputs for both branches
    feat_mlp = torch.tensor(feat, dtype=torch.float32).to(DEVICE)
    feat_vqc = torch.tensor(scaler.transform(pca.transform(feat)), dtype=torch.float32).to(DEVICE)

    # Get predictions
    mlp.eval(); vqc.eval()
    with torch.no_grad():
        mlp_probs = torch.softmax(mlp(feat_mlp), dim=1)[0].cpu().numpy()
        vqc_probs = torch.softmax(vqc(feat_vqc), dim=1)[0].cpu().numpy()

    # Weighted ensemble
    ens_probs = (mlp_probs * 0.6) + (vqc_probs * 0.4)

    return {
        'image'   : img,
        'mlp'     : {'label': class_names[np.argmax(mlp_probs)], 'conf': mlp_probs.max() * 100},
        'vqc'     : {'label': class_names[np.argmax(vqc_probs)], 'conf': vqc_probs.max() * 100},
        'ensemble': {'label': class_names[np.argmax(ens_probs)], 'conf': ens_probs.max() * 100}
    }


# ── Upload & Predict ─────────────────────────────────────────
from google.colab import files
print("📤 Upload image(s) for prediction...")
uploaded = files.upload()

if uploaded:
    for fname in uploaded.keys():
        r = predict_image(fname)

        # Determine verdict from ensemble label
        ens_label_upper = r['ensemble']['label'].upper()
        if 'NON' in ens_label_upper or 'NORMAL' in ens_label_upper:
            verdict = 'NO CANCER'
            color   = '#22c55e'
        else:
            verdict = 'CANCER DETECTED'
            color   = '#ff4444'

        # ── Visualize Result ──────────────────────────────────
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))

        axes[0].imshow(r['image'])
        axes[0].axis('off')
        axes[0].set_title(f'Input: {fname}', fontsize=11)

        result_txt = (
            f"MLP Pred  : {r['mlp']['label'].upper()} ({r['mlp']['conf']:.1f}%)\n"
            f"VQC Pred  : {r['vqc']['label'].upper()} ({r['vqc']['conf']:.1f}%)\n"
            f"{'─'*35}\n"
            f"ENSEMBLE  : {verdict}\n"
            f"Confidence: {r['ensemble']['conf']:.1f}%"
        )

        axes[1].text(
            0.1, 0.5, result_txt, transform=axes[1].transAxes,
            fontsize=13, verticalalignment='center', fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='#f0f4ff', alpha=0.9,
                      edgecolor=color, linewidth=2)
        )
        axes[1].axis('off')
        axes[1].set_title('Detection Result', fontsize=13, fontweight='bold', color=color)
        plt.tight_layout()
        plt.show()